# Excercise - SELECT & FILTER

In [0]:
# /Volumes/workspace/default/sparkhandsonfiles/employee.csv

In [0]:
df = spark.read.format('csv')\
                .option('mode','PERMISSIVE')\
                .option('header',True)\
                .option('inferSchema',True)\
                .load(r'/Volumes/workspace/default/sparkhandsonfiles/employee.csv')
df.show()
df.printSchema()

In [0]:
# 1. Select All Columns: Write a PySpark code to load the CSV and select all columns from the DataFrame.

df.select("*").show() 

In [0]:
# 2. Select Specific Columns: Select only the `ename` and `dept` columns from the DataFrame.
df.select('ename','dept').show()

In [0]:
# 3. Select with Column Renaming: Select the `salary` column and rename it to `monthly_salary`.
df.select('salary').alias('monthly_salary').show()
# Above solution is not renaming the column name

In [0]:
from pyspark.sql.functions import col
df.select(col('salary').alias('monthly_salary')).show()

In [0]:
df.select(df.ename.alias('monthly_sal')).show()

In [0]:
df.selectExpr('salary AS emp_Sal').show()

In [0]:
# 4. Select and Perform an Operation: Select the `salary` column and convert its values from annual to monthly by dividing by 12.

df.select((col('salary')*12).alias('yearly_package')).show()

In [0]:
# 5. Filter by One Condition: Filter the DataFrame to include only employees from the "Tech" department.
df.filter(col('dept') == 'Tech').show()

In [0]:
# 6. Filter by Multiple Conditions: Find employees from the "Finance" department who have a salary greater than 100,000.
df.filter((col('dept') == 'Finance') & (col('salary') > 100000)).show()

In [0]:
df.createOrReplaceTempView('view_employee')

In [0]:
spark.sql("SELECT * FROM view_employee WHERE dept = 'Finance' AND salary > 100000").show()

In [0]:
df.where((col('dept')=='Finance') & (col('salary')>100000)).show()

In [0]:
# 7. Date Filter: Filter employees who joined after January 1, 2015.
df.filter(col('date_of_joining') > '2015-01-01').show()

In [0]:
df.where(col('date_of_joining') > '2015-01-01').show()

In [0]:
# 8. Select and Filter Combination: Select the `eid` and `ename` of all employees in the "HR" department.
df.filter(col('dept') == 'HR').select('eid','ename').show()

In [0]:
# 9. Multiple Conditions with Select: Select `ename`, `dept`, and `date_of_joining` for employees who earn more than 75,000 and work in the "Marketing" department

df.filter((col('salary') > 75000) & (col('dept') == 'Marketing')).select('ename','dept','date_of_joining').show()

In [0]:
# 10. Complex Combination: Select `ename` and `salary`, and filter for employees who joined before 2010 and have a salary less than 80,000.
df.filter((col('date_of_joining') < '2010-01-01') & (col('salary') < 80000)).select('ename','salary').show()

In [0]:
df.where((df.date_of_joining < '2010-01-01') & (df.salary < 80000)).select('ename','salary').show()

In [0]:
# 11. Select Employees with a Salary Over $100,000: Demonstrate how to filter rows in a DataFrame and in SQL where salary is greater than 100,000.
df.filter(df.salary > 100000).show()

In [0]:
spark.sql("SELECT * FROM view_employee WHERE salary > 80000").show()

In [0]:
# 12. Select Employee Names and Departments Where the Department is Not 'Tech': Show how to select and filter employee names and departments excluding those in the 'Tech' department.
df.filter(df.dept != 'Tech').select(df.ename, df.dept).show()

In [0]:
# 13. Calculate the Monthly Salary for Each Employee: Use both DataFrame transformation and SQL to compute the monthly salary by dividing the annual salary by 12.---->Already Done

spark.sql('SELECT salary, salary*12 AS yearly_package FROM view_employee').show()

In [0]:
# 14. Filter Employees Who Joined Before 2015 and Select Their Names and Joining Dates: Illustrate how to filter employees based on their joining date before the year 2015 and select relevant columns.

df.select('ename','date_of_joining').where(df.date_of_joining < '2015-01-01').show()

In [0]:
#  15. Select Employees' Names with 'a' in Their Name: Demonstrate how to filter employees whose names contain the letter 'a'
spark.sql("SELECT * FROM view_employee WHERE ename LIKE '%a%'").show()


In [0]:
df.filter("ename like '%a%'").show()

In [0]:
# 16. Show Employees' Name and a Boolean Column If Salary Is Above Average: Use both DataFrame methods and SQL to add a boolean column indicating whether an employee's salary is above the average salary.
df.withColumn('IsAboveAvg',
              if(df.salary > avg(df.salary), 'yes'),
              else('no'))


In [0]:
# option 1

from pyspark.sql.functions import col,when,avg

avgSal = df.select(avg('salary')).collect()[0][0]

df.withColumn('IsAboveAvg',
              when(df.salary > avgSal,'yes').otherwise('no')).select('ename','salary','IsAboveAvg').show()

In [0]:
Notes = """
Visualizing the whole process

Step 1
df.select(avg("salary"))

Result:

DataFrame
┌───────────┐
│avg(salary)│
├───────────┤
│50000      │
└───────────┘
Step 2
.collect()

Result:

[
    Row(avg(salary)=50000)
]
Step 3
[0]

gets the first row:

Row(avg(salary)=50000)
Step 4
[0]

gets the first column value:

50000

Therefore:

.collect()[0][0]

→ 50000



Why not use filter() to calculate the average?

    You might be thinking of something like:

    df.filter(df.salary > avg(df.salary))

    But there's a conceptual problem.

    filter() is asking:

    Which rows should I keep?

    Whereas avg() is asking:

    What is the aggregate value across the rows?

    So Spark has to know the average before it can decide which rows satisfy the filter.

"""

In [0]:
spark.sql("SELECT ename,salary,salary>AVG(salary) AS IsAboveAvg FROM view_employee").show()

In [0]:
# option 2

spark.sql("SELECT ename,salary, AVG(salary) OVER() AS AvgSal, salary>avgSal AS IsAboveAvg FROM view_employee").show()

In [0]:
# option 3

spark.sql("SELECT ename, salary > (SELECT AVG(salary) FROM view_employee) as above_avg_salary FROM view_employee").show()

In [0]:
#  17. Filter Employees from the 'Finance' Department and Select All Columns: Explain how to filter employees who are in the 'Finance' department.
df.filter(df.dept == 'Finance').show()

In [0]:
# 18. List Employees and the Year of Their Joining: Display how to extract the year from a date column both using DataFrame operations and SQL.

# option 1 
spark.sql("SELECT ename, YEAR(date_of_joining) AS YearOfJoining FROM view_employee").show()

In [0]:
# option 2

from pyspark.sql.functions import year

df.select('ename', year(df.date_of_joining)).show()

In [0]:
# option 3

df.selectExpr("ename", "year(date_of_joining) as joining_year").show()

In [0]:
#  19. Filter for Employees Earning More Than $60,000 and in the 'HR' Department: Show filtering based on multiple conditions involving salary and department.

df.filter((df.salary > 60000) & (df.dept == 'HR')).show()

In [0]:
#  20. Show if Employees Joined in the Last Decade: Calculate using both DataFrame operations and SQL whether employees joined in the last decade based on the current date.

spark.sql("SELECT ename, (YEAR(current_date()) - YEAR(date_of_joining) <= 10) as joined_last_decade FROM view_employee").show()